<a href="https://colab.research.google.com/github/jinhowon/RegressionForHousePrice/blob/main/JinhoWon_SmallGroupRegression2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error

In [ ]:
import kagglehub
kagglehub.login()

path = kagglehub.competition_download('regression-for-house-prices')
print("Path to competition files:", path)

100%|██████████| 192k/192k [00:00<00:00, 72.1MB/s]

Extracting files...
Path to competition files: /root/.cache/kagglehub/competitions/regression-for-house-prices


Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
df = pd.read_csv(path + '/ames/train.csv')
test = pd.read_csv(path + '/ames/test.csv')
df = df[df['GrLivArea'] < 4500]

In [ ]:
X = df.drop(columns=['Id', 'SalePrice'])
y = df['SalePrice']

In [ ]:
num_col = X.select_dtypes(['int', 'float']).columns.tolist()
cat_col = X.select_dtypes('object').columns.tolist()

ord_col = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC',
           'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond']
ohe_col = X[cat_col].drop(columns=ord_col).columns.tolist()

vals = [['Po', 'Fa', 'TA', 'Gd', 'Ex']]



In [ ]:
num = Pipeline([('impute', SimpleImputer()), ('scale', StandardScaler())])
ord = Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                ('ord', OrdinalEncoder(categories=vals*len(ord_col)))])
ohe = Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                ('ohe', OneHotEncoder(handle_unknown='ignore'))])

transformer = make_column_transformer((num, num_col),
                                      (ord, ord_col),
                                      (ohe, ohe_col))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
## spliting data into 75 to train the model and 25% to test
## random state = 42 keeps split the same every run




# baseline: always guess the average training file
#
baseline = np.ones(len(y_test)) * y_train.mean()
print('Baseline RMSE:', np.sqrt(mean_squared_error(y_test, baseline)))

Baseline RMSE: 79207.44851916729


In [ ]:
y_train_log = np.log(y_train)

##predict log(SalePrice) instead of raw price
##House prices are right-skewed (a few very expensive homes), so errors on those homes dominated the plain model. Taking the log makes the target more evenly spread and treats errors as percentages instead of raw dollars.
model = Pipeline([('transform', transformer), ('model', Ridge(alpha=10))])
model.fit(X_train, y_train_log)

##Ridge instead of LinearRegression. One-hot encoding creates many columns, and plain LinearRegression overfit
# convert predictions back to dollars with np.exp
train_preds = np.exp(model.predict(X_train))
test_preds = np.exp(model.predict(X_test))

print('Train RMSE:', np.sqrt(mean_squared_error(y_train, train_preds)))
print('Test RMSE:', np.sqrt(mean_squared_error(y_test, test_preds)))

# Result: test RMSE dropped from ~$28,471 (Regression 1) to ~$19,978, about 30% better, and the train/test gap shrank (less overfitting)

Train RMSE: 16842.981156442038
Test RMSE: 19977.950551306527


In [ ]:
model.fit(X, np.log(y))
##retrain on ALL training data
preds = np.exp(model.predict(test[X.columns]))


In [ ]:
pdf = pd.DataFrame(preds, columns=['SalePrice'])
pdf['Id'] = test.index
pdf = pdf[['Id', 'SalePrice']]

# cap any prediction above the highest price in the training data
pdf.loc[pdf['SalePrice'] > y.max(), 'SalePrice'] = y.max()

pdf.to_csv('final1.csv', index=False)
pdf.head()

,Id,SalePrice
0,0,99797.991748
1,1,195493.689973
2,2,130460.969716
3,3,199025.268843
4,4,149654.717770
